# Madhava Adaptive v5: Numba-JIT Accelerated, Calibrated, Guaranteed

### Three Fixes from v4 Audit:

**1. Floating-point epsilon** — Upper bound now adds EPSILON=1e-5 to absorb float32/64 round-off errors, restoring the zero-violation guarantee.

**2. Calibrated adaptive_keep** — Stage 1 now retains 10-40% of vectors (was 3-20%). Gives Stage 2 (64D) more candidates to find true neighbors.

**3. Numba @njit acceleration** — The critical search loop (projection, bound computation, modulation) is compiled with Numba JIT.

**License:** BSL 1.1 | **Contact:** pay@winnex.ai


In [ ]:
import sys,os,warnings,math,time,random,gc,json,glob
import numpy as np
os.environ['TOKENIZERS_PARALLELISM']='false'; warnings.filterwarnings('ignore')
for pkg,imp in [('faiss-cpu','faiss'),('scikit-learn','sklearn'),('scipy','scipy')]:
    try: __import__(imp)
    except: import subprocess; subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
# Install numba if needed
try: import numba
except: import subprocess; subprocess.check_call([sys.executable,'-m','pip','install','-q','numba'])
import faiss, numba
from numba import njit, prange
SEED=42; np.random.seed(SEED); random.seed(SEED)


In [ ]:
import numpy as np, math, time

# Numba-JIT compiled kernel for the projection + bound computation
@njit(fastmath=True,cache=True)
def _project_and_bound_32d(vectors, proj_matrix, query, errors):
    """32D projection + upper bound for all N vectors. Single pass, no temp arrays."""
    n = vectors.shape[0]
    d_out = proj_matrix.shape[0]  # 32
    d_in = proj_matrix.shape[1]   # 128
    
    # Project query
    q_proj = np.zeros(d_out, dtype=np.float64)
    for k in range(d_out):
        s = 0.0
        for j in range(d_in):
            s += query[j] * proj_matrix[k, j]
        q_proj[k] = s
    
    q_norm_sq = 0.0
    for k in range(d_out):
        q_norm_sq += q_proj[k] * q_proj[k]
    q_res = math.sqrt(max(0.0, 1.0 - q_norm_sq))
    
    # Upper bounds for all vectors
    ub = np.zeros(n, dtype=np.float64)
    for i in range(n):
        pv_dot = 0.0
        for k in range(d_out):
            pv_dot += vectors[i, k] * q_proj[k]
        ub[i] = pv_dot + errors[i] * q_res + 1e-5  # EPSILON for float round-off
    return ub, q_proj, q_res

@njit(fastmath=True,cache=True)
def _refine_64d(vectors_64d_sel, proj_matrix, query, errors_sel, ub_32d_sel, alphas):
    """64D refinement for survivors. Returns modulated scores."""
    m = vectors_64d_sel.shape[0]
    d_out = 64
    d_in = 128
    
    q_proj = np.zeros(d_out, dtype=np.float64)
    for k in range(d_out):
        s = 0.0
        for j in range(d_in):
            s += query[j] * proj_matrix[k, j]
        q_proj[k] = s
    
    q_norm_sq = 0.0
    for k in range(d_out):
        q_norm_sq += q_proj[k] * q_proj[k]
    q_res = math.sqrt(max(0.0, 1.0 - q_norm_sq))
    
    scores = np.zeros(m, dtype=np.float64)
    for i in range(m):
        pv_dot = 0.0
        for k in range(d_out):
            pv_dot += vectors_64d_sel[i, k] * q_proj[k]
        ub_64 = pv_dot + errors_sel[i] * q_res + 1e-5
        scores[i] = ub_32d_sel[i] + alphas[i] * (ub_64 - ub_32d_sel[i])
    return scores

class MadhavaAdaptive:
    def __init__(self,final_topk=500):
        self.full_dim=128; self.base_keep=0.25
        self.max_stage1=150000; self.final_topk=final_topk
        self.rng=np.random.RandomState(43)
        self.vectors=None; self.proj_L={}; self.error={}; self.proj_matrices={}
        self.vectors_32d=None; self.vectors_64d=None  # pre-projected for Numba
    def _make_orthogonal_proj(self,d_out,d_in):
        Q,_=np.linalg.qr(self.rng.randn(d_out,d_in).astype(np.float64).T)
        P=Q[:,:d_out].T.astype(np.float64)
        assert np.abs(P@P.T-np.eye(d_out)).max()<1e-5; return P
    def build(self,vectors):
        t0=time.time(); self.vectors=vectors; self.n_vectors=len(vectors)
        self.norms=np.linalg.norm(vectors,axis=1).astype(np.float64)
        for d in [32,64]:
            P=self._make_orthogonal_proj(d,self.full_dim)
            self.proj_matrices[d]=P
            proj=(vectors.astype(np.float32)@P.T.astype(np.float32)).astype(np.float64)
            self.proj_L[d]=proj
            captured=np.linalg.norm(proj,axis=1)
            self.error[d]=np.sqrt(np.maximum(self.norms**2-captured**2,0))
        # Cache projections for Numba
        self.vectors_32d=self.proj_L[32].copy()
        self.vectors_64d=self.proj_L[64].copy()
        self.build_time=time.time()-t0; return self
    def search(self,q,retprof=False):
        q=q.astype(np.float64).flatten(); p={'n_total':self.n_vectors}
        # Stage 1: 32D via Numba kernel
        ub32, q32, qr32 = _project_and_bound_32d(
            self.vectors_32d, self.proj_matrices[32], q, self.error[32])
        b_range=float(ub32.max()-ub32.min())
        # Calibrated: 10-40% retention
        adaptive_keep=min(0.40,max(0.10,self.base_keep*0.25/max(b_range,0.01)))
        k1=min(max(int(self.n_vectors*adaptive_keep),2000),self.max_stage1,self.n_vectors)
        idx1=np.arange(self.n_vectors) if self.n_vectors<=k1 else np.argpartition(-ub32,k1-1)[:k1]
        p['n1']=len(idx1)
        # Stage 2: 64D refinement via Numba kernel
        e32_sel=self.error[32][idx1]; e64_sel=self.error[64][idx1]
        alphas=1.0/(1.0+np.exp(-(e32_sel-e64_sel)/max(np.mean(e32_sel),1e-9)*0.5))
        scores=_refine_64d(self.vectors_64d[idx1],self.proj_matrices[64],
                          q,e64_sel,ub32[idx1],alphas)
        k2=min(self.final_topk,len(idx1))
        idx2=idx1[np.argpartition(-scores,k2-1)[:k2]]
        # Exact cosine on survivors (in original 128D)
        cos=self.vectors[idx2].astype(np.float64)@q.astype(np.float64)
        top=idx2[np.argsort(-cos)][:FINAL_K]
        if retprof: return top,p
        return top
    def bound_violations(self,q):
        q=q.astype(np.float64).flatten(); qn=np.linalg.norm(q)
        tc=self.vectors.astype(np.float64)@q; v={}
        for d in [32,64]:
            P=self.proj_matrices[d]
            qd=(q.astype(np.float32)@P.T.astype(np.float32)).astype(np.float64)
            ub=self.proj_L[d]@qd+self.error[d]*math.sqrt(max(0,qn**2-np.linalg.norm(qd)**2))+1e-5
            v[f'{d}D']=int(np.sum(tc>ub+1e-9))
        return v,self.n_vectors
FINAL_K=10; NDCG_K=10


In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
print('Loading SBERT...')
embdr=SentenceTransformer('all-MiniLM-L6-v2',device='cpu')
NEWS_PATH=None
for p in sorted(glob.glob('/kaggle/input/**/*.json',recursive=True)):
    if 'news' in p.lower() and 'category' in p.lower(): NEWS_PATH=p; break
if NEWS_PATH is None:
    for p in ['../input/news-category-dataset/News_Category_Dataset_v3.json','/kaggle/input/news-category-dataset/News_Category_Dataset_v3.json','News_Category_Dataset_v3.json']:
        if os.path.exists(p): NEWS_PATH=p; break
records=[]
with open(NEWS_PATH) as f:
    for line in f:
        if line.strip(): records.append(json.loads(line))
df=pd.DataFrame(records).dropna().reset_index(drop=True)
texts=(df['headline'].fillna('')+' '+df.get('short_description',df.get('description','')).fillna('')).tolist()
N_ENCODE=min(20000,len(texts))
print(f'Encoding {N_ENCODE} texts...')
embs=embdr.encode(texts[:N_ENCODE],convert_to_tensor=False,show_progress_bar=True,normalize_embeddings=True)
embs=np.array(embs).astype(np.float32)
QJL_MATRIX=(np.random.RandomState(42).randn(384,128)/np.sqrt(128)).astype(np.float32)
E=(embs.astype(np.float32)@QJL_MATRIX)
E/=np.linalg.norm(E,axis=1,keepdims=True)
print(f'128D shape: {E.shape}')


In [ ]:
NQ=200
rng=np.random.RandomState(42)
query_idx=rng.choice(np.arange(len(E)),NQ,replace=False)
Q=E[query_idx]
all_cats=df['category'].values
def rel_set(q_idx):
    pos=query_idx[q_idx]; qcat=df['category'].iloc[pos]
    return {i for i in range(len(E)) if all_cats[i]==qcat}
def ndcg_binary(ranked,rs,k=10):
    dcg=sum(1.0/math.log2(j+2) for j,idx in enumerate(ranked[:k]) if int(idx) in rs)
    nrel=min(len(rs),k); idcg=sum(1.0/math.log2(j+2) for j in range(nrel))
    return dcg/idcg if idcg>0 else 0
def recall_binary(ranked,rs,k=10):
    if not rs: return 0
    return sum(1 for i in ranked[:k] if int(i) in rs)/min(len(rs),k)

fi=faiss.IndexFlatIP(128); fi.add(E)
hnsw_idx=faiss.IndexHNSWFlat(128,32); hnsw_idx.hnsw.efConstruction=200; hnsw_idx.add(E)
nlist=min(int(math.sqrt(len(E))),256)
quant=faiss.IndexFlatIP(128)
ivf_idx=faiss.IndexIVFFlat(quant,128,nlist,faiss.METRIC_INNER_PRODUCT)
ivf_idx.train(E); ivf_idx.add(E)

print(f"{'Method':>28} {'NDCG@10':>10} {'Recall@10':>10} {'Lat(ms)':>10} {'Build':>10} {'Viol':>6}")
print(f"{'-'*75}")

# FlatIP 128D
ft,fnd,fr=[],[],[]
for qi in range(NQ):
    qv=Q[qi:qi+1]; rs=rel_set(qi)
    t0=time.time(); D,I=fi.search(qv,NDCG_K); ft.append((time.time()-t0)*1000)
    fnd.append(ndcg_binary(I[0],rs)); fr.append(recall_binary(I[0],rs))
print(f"{'FlatIP(128D)':>28} {np.mean(fnd):>10.4f} {np.mean(fr):>10.4f} {np.mean(ft):>10.3f} {'N/A':>10} {'-':>6}")

# HNSW ef=128
hnsw_idx.hnsw.efSearch=128
ht,hn,hr=[],[],[]
for qi in range(NQ):
    qv=Q[qi:qi+1]; rs=rel_set(qi)
    t0=time.time(); D,I=hnsw_idx.search(qv,NDCG_K); ht.append((time.time()-t0)*1000)
    hn.append(ndcg_binary(I[0],rs)); hr.append(recall_binary(I[0],rs))
print(f"{'HNSW(ef=128) 128D':>28} {np.mean(hn):>10.4f} {np.mean(hr):>10.4f} {np.mean(ht):>10.3f} {'N/A':>10} {'-':>6}")

# IVF nprobe=20
ivf_idx.nprobe=20
it,ind,ir=[],[],[]
for qi in range(NQ):
    qv=Q[qi:qi+1]; rs=rel_set(qi)
    t0=time.time(); D,I=ivf_idx.search(qv,NDCG_K); it.append((time.time()-t0)*1000)
    ind.append(ndcg_binary(I[0],rs)); ir.append(recall_binary(I[0],rs))
print(f"{'IVF(nprobe=20) 128D':>28} {np.mean(ind):>10.4f} {np.mean(ir):>10.4f} {np.mean(it):>10.3f} {'N/A':>10} {'-':>6}")

for ftopk in [200,500,1000]:
    mad=MadhavaAdaptive(final_topk=ftopk); mad.build(E)
    # Warmup Numba cache
    mad.search(Q[0])
    mt,mnd,mr=[],[],[]
    for qi in range(NQ):
        qv=Q[qi]; rs=rel_set(qi)
        t0=time.time(); top=mad.search(qv); mt.append((time.time()-t0)*1000)
        mnd.append(ndcg_binary(top,rs)); mr.append(recall_binary(top,rs))
    viols={}
    for qi in range(min(10,NQ)):
        v,_=mad.bound_violations(Q[qi])
        for kk,vv in v.items(): viols[kk]=max(viols.get(kk,0),vv)
    vio='OK' if all(v==0 for v in viols.values()) else '!'
    print(f"{'Madhava(ftopk='+str(ftopk)+')':>28} {np.mean(mnd):>10.4f} {np.mean(mr):>10.4f} {np.mean(mt):>10.3f} {mad.build_time:>7.3f}s {vio:>6}")
    # Save final config results
    if ftopk==500:
        res={'config':{'N':len(E),'dim':128,'queries':NQ,'dataset':'news_category'}}
        res['flatip']={'ndcg':float(np.mean(fnd)),'recall':float(np.mean(fr)),'lat_ms':float(np.mean(ft))}
        res['hnsw_128']={'ndcg':float(np.mean(hn)),'recall':float(np.mean(hr)),'lat_ms':float(np.mean(ht))}
        res['ivf_20']={'ndcg':float(np.mean(ind)),'recall':float(np.mean(ir)),'lat_ms':float(np.mean(it))}
        res['madhava']={'ndcg':float(np.mean(mnd)),'recall':float(np.mean(mr)),'lat_ms':float(np.mean(mt)),'build_s':mad.build_time}
        with open('madhava_v5_results.json','w') as f: json.dump(res,f,indent=2)
        print(f"{'Saved results to madhava_v5_results.json':>60}")